# POOP — Programming Elements & OOP (9569)

Revision / content-lookup notebook for **S2A Programming Elements** and **S2B Object-Oriented Programming**.

## Contents
1. [Programming elements — validation, files, filter/sort](#1)
2. [Mod-11 check digit](#2)
3. [OOP — encapsulation, inheritance, polymorphism](#3)
4. [Exam lookup cheat-sheet](#4)

**How to use:** skim the markdown for definitions; run the code cells when you need a working pattern. Keep syntax on the exam cheatsheet/docs; memorise the *logic* here.


<a id="1"></a>
## 1. Programming elements

Typical Paper 2 pattern: **validate → read file → store in a structure → filter/sort → output**.

### Validation types (know the names)
| Type | Question to ask |
|------|-----------------|
| Presence | Is the field non-empty? |
| Length | Is length exactly / at most / at least *n*? |
| Range | Is the number between min and max (or `> 0`)? |
| Format / type | Digits only? Letters + digits in fixed positions? Convertible to `float`? |
| Check digit | Does a calculated digit match (e.g. mod-11)? |

### Exam habits
- Prefer `with open(...)` so the file always closes.
- `next(file)` / `next(reader)` skips a header row cleanly.
- `try/except` around file open and around risky conversions (`float`, `int`).
- Store records as **dicts** (or tuples) so later tasks stay readable.


### SKU format check
Length + format validation. Pattern below: length 4, first two alphabetic, last two digits.


In [ ]:
def is_valid_sku(sku):
    if len(sku) == 4 and sku[:2].isalpha() and sku[2:].isdigit():
        return True
    else:
        return False

### Load from text file
Pipeline: open → skip header → `strip().split(',')` → validate fields → append dict → return list.

Invalid rows should be skipped (optionally with a short message). Missing file should not crash the program.


In [ ]:
def load_products(filename):
    database = []
    try:
        with open(filename, 'r') as file:
            next(file)
            for line in file:
                line = line.strip().split(',')
                try:
                    if len(line) == 3 and is_valid_sku(line[0]) and float(line[2]) > 0: 
                        database.append({
                            "sku": line[0],
                            "name": line[1],
                            "price": float(line[2])
                        })
                except:
                    print("price not a float")
    except: 
        print("no file")
    return database   

### Filter and sort
Keep only records meeting a condition, then sort.

**Two patterns — pick by what the question asks for:**

- `(price, name)` tuple: fine when output only needs those fields; `sorted()` sorts by price automatically.
- Full dict: use when you need `sku` or other fields; `sorted(data, key=lambda p: p["price"])`.

The function below uses the tuple pattern. To return full records, replace `data.append((i.get('price'), i.get('name')))` with `data.append(i)` and add `key=lambda p: p["price"]` to the `sorted()` call.


In [ ]:
def filter_by_max_price(products, max_price):
    data = []
    for i in products:
        if float(i.get("price")) <= max_price:
            data.append((i.get('price'), i.get('name')))
    return sorted(data)

<a id="2"></a>
## 2. Mod-11 check digit

Used for **data validation** of ID codes (ISBN-style).

### Algorithm (generate)
1. Convert code to string of digits (no check digit yet).
2. Multiply each digit by a weight. Common weighting: leftmost weight = `len(code) + 1`, then decrease by 1 down to **2**.
3. `remainder = product_sum % 11`
4. `check = 11 - remainder`
5. Special cases: **10 → `'X'`**, **11 → `'0'`** (when remainder was 0).

### Verify
Recalculate the check digit from `code[:-1]` and compare to `code[-1]` (normalise case for `X`).

### Functions in this notebook
- `mod11_generate_check(code)` → check character  
- `add_check_digit(code)` → original + check  
- `verify_check_digit(code)` → `True` / `False`


In [ ]:
def mod11_generate_check(code):
    #data validation and cleaning
    try:
        #code in int form is converted to str, if in str, no change
        code = str(code)
        i = len(code)
        product_sum = 0
        for n in code:
            product_sum += int(n) * (i+1) 
            i -= 1
        check_digit = str(11 - (product_sum % 11))
        if check_digit == '10':
            check_digit = 'X'
        elif check_digit == '11':
            check_digit = '0'
        return check_digit
    except:
        print("invalid data type. please use an integer or string of integers")
        return -1

def add_check_digit(code):
    try:
        return str(code) + mod11_generate_check(code)
    except:
        print("invalid data type. please use an integer or string of integers")
        return -1
        
def verify_check_digit(code):
    try:
        if mod11_generate_check(code[:-1]) == code[-1].upper():
            return True
        else:
            return False
    except: 
        return False

In [ ]:
test = add_check_digit(1892839)
verify_check_digit(test)

<a id="3"></a>
## 3. OOP (all-in-one)

### Core principles (9569)
| Principle | Meaning | In this notebook |
|-----------|---------|------------------|
| **Encapsulation** | Hide data (`__attr`); access via getters/setters | `__id`, `__title`; no public setter for ID |
| **Inheritance** | Child *is-a* parent; reuse attributes/methods | `DigitalResource(Resource)` |
| **Polymorphism** | Same method name, different behaviour by class | both implement `describe()`; loop calls the right one |
| **Override** | Child replaces parent method of the same name | `DigitalResource.describe` |
| **`super()`** | Call parent constructor/method from child | `super().__init__(ID, title)` |

### Name mangling trap
`self.__id` inside class `A` becomes `_A__id`. A child must **not** read parent `__` attributes directly — use **getters** (or public API). That is why `DigitalResource.describe` uses `self.get_id()` / `self.get_title()`.

### Constructor chaining
Child `__init__` should call `super().__init__(...)` with the parent’s parameters, then set child-only attributes (e.g. `__filesize`).


### Parent class — `Resource`
Blueprint for a general resource: private ID/title, getters, title setter only, `describe()`.


In [ ]:
class Resource:
    def __init__(self, ID, title):
        self.__id = ID
        self.__title = title
        
    def get_id(self):
        return self.__id
    
    def get_title(self):
        return self.__title
        
    def set_title(self, title):
        self.__title = title
    
    def describe(self):
        return f"Resource {self.__id}: {self.__title}"
        

### Child class — `DigitalResource`
Extends `Resource` with `__filesize`, chains via `super()`, **overrides** `describe()`.

Inherited getters/setters are available automatically — no need to re-wrap them unless behaviour changes.


In [ ]:
class DigitalResource(Resource):
    def __init__(self, ID, title, filesize):
        super().__init__(ID, title)
        self.__filesize = int(filesize)
        
    def describe(self):
        return f"Digital Resource {self.get_id()}: {self.get_title()} ({self.__filesize} MB)"

### Polymorphism test
Put parent and child **instances** in one list. Loop and call `describe()` — each object responds with its own version.

This is the classic exam demonstration of polymorphism.


In [ ]:
parent = Resource(6767, "im a father!")
child = DigitalResource(6969, "newborn", 89)
thing = [parent,child]
for i in thing:
    print(i.describe())

<a id="4"></a>
## 4. Exam lookup cheat-sheet

### Programming elements — minimum recall
```text
validate (length / format / range / type / check digit)
with open(path) as f:
    next(f)                    # skip header
    for line in f:
        parts = line.strip().split(',')
        # validate → append dict
try/except around open and float/int conversions
filter with a loop (or comprehension) → sorted(..., key=...)
```

### Mod-11 — minimum recall
```text
weights: len+1 … 2
check = 11 - (sum % 11)
10 → X ; 11 → 0
verify: regenerate from all-but-last, compare to last
```

### OOP — minimum recall
```text
class Child(Parent):
    def __init__(self, ...):
        super().__init__(...)
        self.__extra = ...
    def describe(self):           # override
        ... self.get_...( ) ...   # use getters for parent private data

objs = [Parent(...), Child(...)]
for o in objs:
    o.describe()                  # polymorphism
```

### Theory one-liners
- **Encapsulation:** protect state; controlled access.  
- **Inheritance:** share/extend an existing class (is-a).  
- **Polymorphism:** one interface, many implementations.  
- **Composition** (contrast): object *has-a* another object as an attribute — not shown in code above, but examiners may ask is-a vs has-a.
